# Sobel vs Canny para Detecção de Trincas em Concreto

Este notebook organiza o script de **Sobel vs Canny** em formato de aula para Google Colab, com processamento em lote de imagens de obra.

O objetivo é comparar dois métodos clássicos de detecção de bordas:

- **Sobel**, baseado em gradientes nas direções X e Y;
- **Canny**, baseado em suavização, gradiente, supressão de não máximos e limiares por histerese.

A aplicação didática é o realce de **bordas finas**, como trincas, fissuras, juntas e descontinuidades visuais em imagens de concreto.

**Autora:** Adriana Rolim, coordenação  
**Versão do script:** 2.0 — Otimizado para Colab + Processamento em Lote  
**Data:** Agosto de 2025

## 1. O que este notebook faz?

O notebook executa o seguinte fluxo:

```text
Imagem da obra
→ redimensionamento, se necessário
→ conversão para escala de cinza
→ suavização gaussiana
→ detecção de bordas com Sobel
→ detecção de bordas com Canny
→ geração de overlays
→ comparação visual
→ relatório CSV consolidado
```

Para cada imagem, ele cria uma subpasta com os produtos gerados.

## 2. Sobel: ideia central

O operador **Sobel** calcula a variação de intensidade dos pixels em duas direções:

- **Sobel X:** destaca variações horizontais;
- **Sobel Y:** destaca variações verticais.

Depois, as duas respostas são combinadas para formar a **magnitude do gradiente**.

Em imagens de concreto, isso ajuda a destacar regiões onde há mudança brusca de intensidade, como trincas, fissuras, bordas de elementos, juntas e texturas marcadas.

## 3. Canny: ideia central

O detector **Canny** é um método mais completo para detecção de bordas.

Ele combina:

1. suavização da imagem;
2. cálculo do gradiente;
3. supressão de não máximos;
4. dois limiares;
5. rastreamento de bordas por histerese.

Na prática, o Canny costuma gerar bordas mais finas e limpas, mas depende muito dos limiares configurados.

## 4. Diferença prática entre Sobel e Canny

| Aspecto | Sobel | Canny |
|---|---|---|
| Tipo | Operador de gradiente | Detector completo de bordas |
| Resultado | Intensidade/magnitude de borda | Mapa binário de bordas |
| Sensibilidade a ruído | Maior | Menor, se bem parametrizado |
| Controle | Kernel e limiarização | Dois limiares e suavização |
| Uso didático | Excelente para entender gradientes | Excelente para comparar bordas finais |

Nenhum dos dois métodos, isoladamente, “detecta trincas” semanticamente. Eles realçam bordas que podem ou não corresponder a trincas reais.

## 5. Importação das bibliotecas

Execute esta célula para carregar as bibliotecas necessárias.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
from google.colab import drive
import pandas as pd

## 6. Configuração do Google Colab

O notebook espera a seguinte estrutura no Google Drive:

```text
MyDrive/
└── Python_VC/
    ├── fotos_obra/
    └── output_sobel_canny/
```

As imagens originais devem estar dentro da pasta `fotos_obra`.

In [ ]:
# Montar o Google Drive
drive.mount('/content/drive')

# Configurações - Mantendo a mesma estrutura do projeto
PASTA_BASE = '/content/drive/MyDrive/Python_VC'
PASTA_ENTRADA = os.path.join(PASTA_BASE, "fotos_obra")
PASTA_SAIDA = os.path.join(PASTA_BASE, "output_sobel_canny")

print("Configuração de pastas carregada.")
print(f"Pasta base: {PASTA_BASE}")
print(f"Pasta de entrada: {PASTA_ENTRADA}")
print(f"Pasta de saída: {PASTA_SAIDA}")

## 7. Parâmetros do algoritmo

Os parâmetros abaixo controlam a suavização, a sensibilidade do Canny, o kernel Sobel e a limpeza morfológica.

In [ ]:
# Parâmetros do algoritmo
PARAMS_SOBEL_CANNY = {
    'ksize_sobel': 3,                    # Tamanho do kernel Sobel (3 para trincas finas)
    'gauss_ksize': (3, 3),               # Suavização Gaussiana
    'gauss_sigma': 0,                    # Sigma automático
    'canny_thresh1': 50,                 # Threshold inferior Canny
    'canny_thresh2': 150,                # Threshold superior Canny
    'redimensionar_max': 1200,           # Largura máxima para redimensionamento
    'morph_kernel_size': 3               # Tamanho do kernel para limpeza morfológica
}

PARAMS_SOBEL_CANNY

## 8. Verificação rápida da pasta de entrada

Esta célula verifica se a pasta `fotos_obra` existe e se há imagens nela.

In [ ]:
def verificar_pasta_entrada():
    """Verifica se a pasta de entrada existe e contém imagens."""
    print("🔍 VERIFICANDO PASTA DE ENTRADA")
    print(f"📂 Entrada: {PASTA_ENTRADA}")

    if not os.path.exists(PASTA_ENTRADA):
        print(f"❌ Pasta de entrada não encontrada: {PASTA_ENTRADA}")
        return False

    extensoes_imagem = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif')
    arquivos_imagem = [
        f for f in os.listdir(PASTA_ENTRADA)
        if f.lower().endswith(extensoes_imagem)
    ]

    if not arquivos_imagem:
        print(f"❌ Nenhuma imagem encontrada em: {PASTA_ENTRADA}")
        return False

    print(f"✅ Imagens encontradas: {len(arquivos_imagem)}")
    for img in arquivos_imagem[:10]:
        print(f"   - {img}")
    if len(arquivos_imagem) > 10:
        print(f"   ... e mais {len(arquivos_imagem) - 10} imagens")

    return True

verificar_pasta_entrada()

## 9. Funções auxiliares

Estas funções criam diretórios, redimensionam imagens e salvam resultados.

In [ ]:
# ======== FUNÇÕES AUXILIARES ========
def criar_diretorio_se_nao_existir(caminho):
    """Cria diretório se não existir."""
    os.makedirs(caminho, exist_ok=True)


def redimensionar_imagem(imagem, largura_max):
    """Redimensiona imagem mantendo proporção."""
    h, w = imagem.shape[:2]

    if largura_max and largura_max > 0 and w > largura_max:
        ratio = largura_max / w
        nova_largura = largura_max
        nova_altura = int(h * ratio)
        imagem_redimensionada = cv2.resize(imagem, (nova_largura, nova_altura))
        print(f"   📐 Redimensionada: {w}x{h} → {nova_largura}x{nova_altura}")
        return imagem_redimensionada

    return imagem


def salvar_resultado(caminho, imagem):
    """Salva imagem com verificação de diretório."""
    criar_diretorio_se_nao_existir(os.path.dirname(caminho))
    cv2.imwrite(caminho, imagem)

## 10. Processamento de uma imagem com Sobel e Canny

Esta função aplica o fluxo completo em uma imagem:

1. carrega a imagem;
2. redimensiona, se necessário;
3. converte para escala de cinza;
4. aplica suavização gaussiana;
5. calcula Sobel X, Sobel Y e magnitude;
6. binariza a magnitude com Otsu;
7. limpa a máscara Sobel;
8. aplica Canny;
9. cria overlays;
10. salva os resultados e retorna estatísticas.

In [ ]:
def processar_imagem_sobel_canny(caminho_imagem, pasta_saida, params):
    """Processa uma imagem com Sobel e Canny para detecção de trincas."""

    nome_arquivo = os.path.basename(caminho_imagem)
    nome_base = os.path.splitext(nome_arquivo)[0]

    print(f"🔍 Processando: {nome_arquivo}")

    # Carregar imagem
    img_bgr = cv2.imread(caminho_imagem)
    if img_bgr is None:
        print(f"❌ Erro ao carregar: {caminho_imagem}")
        return None

    # Redimensionar se necessário
    img_bgr = redimensionar_imagem(img_bgr, params['redimensionar_max'])

    # Converter para escala de cinza
    cinza = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Aplicar suavização Gaussiana
    if params['gauss_ksize'] != (0, 0):
        cinza_suavizado = cv2.GaussianBlur(cinza, params['gauss_ksize'], params['gauss_sigma'])
    else:
        cinza_suavizado = cinza.copy()

    # ======== DETECÇÃO SOBEL ========
    sobel_x = cv2.Sobel(cinza_suavizado, cv2.CV_64F, 1, 0, ksize=params['ksize_sobel'])
    sobel_y = cv2.Sobel(cinza_suavizado, cv2.CV_64F, 0, 1, ksize=params['ksize_sobel'])

    # Magnitude do gradiente
    magnitude = np.hypot(sobel_x, sobel_y)
    magnitude_normalizada = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # Binarização com Otsu
    _, bordas_sobel = cv2.threshold(magnitude_normalizada, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Limpeza morfológica
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (params['morph_kernel_size'], params['morph_kernel_size']))
    bordas_sobel_limpas = cv2.morphologyEx(bordas_sobel, cv2.MORPH_OPEN, kernel)

    # Overlay Sobel
    overlay_sobel = img_bgr.copy()
    overlay_sobel[bordas_sobel_limpas > 0] = (0, 255, 255)  # Ciano
    overlay_sobel = cv2.addWeighted(img_bgr, 0.7, overlay_sobel, 0.3, 0)

    # ======== DETECÇÃO CANNY ========
    bordas_canny = cv2.Canny(cinza_suavizado, params['canny_thresh1'], params['canny_thresh2'])

    # Overlay Canny
    overlay_canny = img_bgr.copy()
    overlay_canny[bordas_canny > 0] = (0, 0, 255)  # Vermelho
    overlay_canny = cv2.addWeighted(img_bgr, 0.7, overlay_canny, 0.3, 0)

    # ======== SALVAR RESULTADOS ========
    pasta_imagem = os.path.join(pasta_saida, nome_base)
    criar_diretorio_se_nao_existir(pasta_imagem)

    # Salvar resultados Sobel
    salvar_resultado(os.path.join(pasta_imagem, f"{nome_base}_sobel_x.png"), cv2.convertScaleAbs(sobel_x))
    salvar_resultado(os.path.join(pasta_imagem, f"{nome_base}_sobel_y.png"), cv2.convertScaleAbs(sobel_y))
    salvar_resultado(os.path.join(pasta_imagem, f"{nome_base}_sobel_magnitude.png"), magnitude_normalizada)
    salvar_resultado(os.path.join(pasta_imagem, f"{nome_base}_sobel_bordas.png"), bordas_sobel_limpas)
    salvar_resultado(os.path.join(pasta_imagem, f"{nome_base}_sobel_overlay.png"), overlay_sobel)

    # Salvar resultados Canny
    salvar_resultado(os.path.join(pasta_imagem, f"{nome_base}_canny_bordas.png"), bordas_canny)
    salvar_resultado(os.path.join(pasta_imagem, f"{nome_base}_canny_overlay.png"), overlay_canny)

    # Salvar imagem original redimensionada
    salvar_resultado(os.path.join(pasta_imagem, f"{nome_base}_original.png"), img_bgr)

    # ======== GERAR RELATÓRIO ========
    pixels_sobel = np.sum(bordas_sobel_limpas > 0)
    pixels_canny = np.sum(bordas_canny > 0)
    pixels_totais = bordas_sobel_limpas.size

    estatisticas = {
        'arquivo': nome_arquivo,
        'pixels_sobel': pixels_sobel,
        'pixels_canny': pixels_canny,
        'percentual_sobel': (pixels_sobel / pixels_totais) * 100,
        'percentual_canny': (pixels_canny / pixels_totais) * 100,
        'diferenca_percentual': abs((pixels_sobel - pixels_canny) / pixels_totais) * 100,
        'largura': img_bgr.shape[1],
        'altura': img_bgr.shape[0]
    }

    return estatisticas

## 11. Visualização comparativa

Esta função mostra a comparação didática entre:

- imagem original;
- magnitude Sobel;
- bordas Sobel;
- overlay Sobel;
- bordas Canny;
- overlay Canny;
- comparação Sobel × Canny.

In [ ]:
def visualizar_resultados_comparativos(caminho_imagem, params):
    """Gera visualização comparativa para uma imagem."""

    img_bgr = cv2.imread(caminho_imagem)
    if img_bgr is None:
        return

    img_bgr = redimensionar_imagem(img_bgr, params['redimensionar_max'])
    cinza = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    if params['gauss_ksize'] != (0, 0):
        cinza = cv2.GaussianBlur(cinza, params['gauss_ksize'], params['gauss_sigma'])

    # Processar para visualização
    sobel_x = cv2.Sobel(cinza, cv2.CV_64F, 1, 0, ksize=params['ksize_sobel'])
    sobel_y = cv2.Sobel(cinza, cv2.CV_64F, 0, 1, ksize=params['ksize_sobel'])
    magnitude = np.hypot(sobel_x, sobel_y)
    magnitude_norm = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, bordas_sobel = cv2.threshold(magnitude_norm, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    bordas_canny = cv2.Canny(cinza, params['canny_thresh1'], params['canny_thresh2'])

    # Criar visualização
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    # Linha 1: Original e Sobel
    axes[0, 0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')

    axes[0, 1].imshow(magnitude_norm, cmap='gray')
    axes[0, 1].set_title('Sobel - Magnitude')
    axes[0, 1].axis('off')

    axes[0, 2].imshow(bordas_sobel, cmap='gray')
    axes[0, 2].set_title('Sobel - Bordas (Otsu)')
    axes[0, 2].axis('off')

    overlay_sobel_viz = img_bgr.copy()
    overlay_sobel_viz[bordas_sobel > 0] = (0, 255, 255)
    overlay_sobel_viz = cv2.addWeighted(img_bgr, 0.7, overlay_sobel_viz, 0.3, 0)
    axes[0, 3].imshow(cv2.cvtColor(overlay_sobel_viz, cv2.COLOR_BGR2RGB))
    axes[0, 3].set_title('Sobel - Overlay')
    axes[0, 3].axis('off')

    # Linha 2: Canny e comparação
    axes[1, 0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[1, 0].set_title('Original')
    axes[1, 0].axis('off')

    axes[1, 1].imshow(bordas_canny, cmap='gray')
    axes[1, 1].set_title(f'Canny (T1={params["canny_thresh1"]}, T2={params["canny_thresh2"]})')
    axes[1, 1].axis('off')

    overlay_canny_viz = img_bgr.copy()
    overlay_canny_viz[bordas_canny > 0] = (0, 0, 255)
    overlay_canny_viz = cv2.addWeighted(img_bgr, 0.7, overlay_canny_viz, 0.3, 0)
    axes[1, 2].imshow(cv2.cvtColor(overlay_canny_viz, cv2.COLOR_BGR2RGB))
    axes[1, 2].set_title('Canny - Overlay')
    axes[1, 2].axis('off')

    # Comparação lado a lado
    comparacao = np.hstack([bordas_sobel, bordas_canny])
    axes[1, 3].imshow(comparacao, cmap='gray')
    axes[1, 3].set_title('Comparação: Sobel (esq) vs Canny (dir)')
    axes[1, 3].axis('off')

    plt.suptitle(f'Análise de Trincas - {os.path.basename(caminho_imagem)}', fontsize=16)
    plt.tight_layout()
    plt.show()

## 12. Função principal de processamento em lote

Esta função percorre todas as imagens da pasta `fotos_obra`, exibe a visualização comparativa e salva os resultados.

In [ ]:
def executar_analise_sobel_canny():
    """Função principal para executar análise Sobel vs Canny."""

    print("=== DETECÇÃO DE TRINCAS - SOBEL vs CANNY ===")
    print(f"📁 Pasta base: {PASTA_BASE}")
    print(f"📂 Entrada: {PASTA_ENTRADA}")
    print(f"📂 Saída: {PASTA_SAIDA}")
    print(f"⚙️ Parâmetros: {PARAMS_SOBEL_CANNY}")
    print("-" * 50)

    # Verificar se pasta de entrada existe
    if not os.path.exists(PASTA_ENTRADA):
        print(f"❌ Pasta de entrada não encontrada: {PASTA_ENTRADA}")
        return None

    # Criar pasta de saída
    criar_diretorio_se_nao_existir(PASTA_SAIDA)

    # Listar imagens
    extensoes_imagem = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif')
    arquivos_imagem = [
        f for f in os.listdir(PASTA_ENTRADA)
        if f.lower().endswith(extensoes_imagem)
    ]

    if not arquivos_imagem:
        print(f"❌ Nenhuma imagem encontrada em: {PASTA_ENTRADA}")
        return None

    print(f"📊 Encontradas {len(arquivos_imagem)} imagens para processamento")

    # Lista para estatísticas
    todas_estatisticas = []

    # Processar cada imagem
    for i, arquivo_imagem in enumerate(arquivos_imagem, 1):
        caminho_imagem = os.path.join(PASTA_ENTRADA, arquivo_imagem)

        print(f"
[{i}/{len(arquivos_imagem)}] Processando: {arquivo_imagem}")

        # Visualização rápida
        visualizar_resultados_comparativos(caminho_imagem, PARAMS_SOBEL_CANNY)

        # Processamento completo
        estatisticas = processar_imagem_sobel_canny(
            caminho_imagem,
            PASTA_SAIDA,
            PARAMS_SOBEL_CANNY
        )

        if estatisticas:
            todas_estatisticas.append(estatisticas)
            print(
                f"✅ Concluído: "
                f"{estatisticas['pixels_sobel']} px Sobel, "
                f"{estatisticas['pixels_canny']} px Canny"
            )

    # Gerar relatório consolidado
    if todas_estatisticas:
        df_estatisticas = pd.DataFrame(todas_estatisticas)
        caminho_relatorio = os.path.join(PASTA_SAIDA, "relatorio_estatisticas.csv")
        df_estatisticas.to_csv(caminho_relatorio, index=False)

        print(f"
📈 RELATÓRIO FINAL:")
        print(f"📊 Total de imagens processadas: {len(todas_estatisticas)}")
        print(f"📋 Média Sobel: {df_estatisticas['percentual_sobel'].mean():.2f}%")
        print(f"📋 Média Canny: {df_estatisticas['percentual_canny'].mean():.2f}%")
        print(f"📁 Relatório salvo em: {caminho_relatorio}")
        print(f"
🎯 Processamento concluído! Resultados em: {PASTA_SAIDA}")

        return df_estatisticas

    print(f"
⚠️ Nenhuma imagem foi processada com sucesso.")
    return None

## 13. Execução

Execute esta célula para processar todas as imagens da pasta `fotos_obra`.

In [ ]:
df_estatisticas = executar_analise_sobel_canny()

## 14. Visualizar relatório consolidado

Depois de executar o processamento, visualize aqui a tabela consolidada.

In [ ]:
if 'df_estatisticas' in globals() and isinstance(df_estatisticas, pd.DataFrame):
    display(df_estatisticas)
else:
    print("Execute primeiro a célula de processamento.")

## 15. Como ajustar os parâmetros

### Canny detectando bordas demais

Aumente os limiares:

```python
PARAMS_SOBEL_CANNY['canny_thresh1'] = 80
PARAMS_SOBEL_CANNY['canny_thresh2'] = 200
```

### Canny perdendo trincas finas

Reduza os limiares:

```python
PARAMS_SOBEL_CANNY['canny_thresh1'] = 30
PARAMS_SOBEL_CANNY['canny_thresh2'] = 100
```

### Muito ruído na imagem

Aumente a suavização:

```python
PARAMS_SOBEL_CANNY['gauss_ksize'] = (5, 5)
```

### Sobel realçando muita textura

Aumente a limpeza morfológica:

```python
PARAMS_SOBEL_CANNY['morph_kernel_size'] = 5
```

In [ ]:
# Célula opcional para ajustes antes de reexecutar

# PARAMS_SOBEL_CANNY['canny_thresh1'] = 30
# PARAMS_SOBEL_CANNY['canny_thresh2'] = 100
# PARAMS_SOBEL_CANNY['gauss_ksize'] = (5, 5)
# PARAMS_SOBEL_CANNY['morph_kernel_size'] = 5

PARAMS_SOBEL_CANNY

## 16. Interpretação técnica dos resultados

Na análise das imagens, observe:

1. **O Sobel destacou textura demais?**  
   Isso pode acontecer em concreto rugoso ou com ruído.

2. **O Canny perdeu trincas finas?**  
   Os limiares podem estar altos demais.

3. **As bordas detectadas fazem sentido visualmente?**  
   Nem toda borda é uma trinca.

4. **Os overlays estão coerentes com a inspeção visual?**  
   Use os overlays para validar rapidamente o comportamento dos métodos.

5. **O percentual de borda está muito alto?**  
   Pode indicar ruído, textura excessiva ou limiares inadequados.

Este notebook deve ser entendido como uma etapa de **realce e comparação de bordas**, não como diagnóstico automático de patologias.

## 17. Fluxo recomendado em visão computacional para trincas

Um fluxo mais completo para inspeção de trincas em concreto pode ser:

```text
Imagem original
→ análise exploratória
→ redimensionamento
→ redução de ruído
→ CLAHE
→ Sobel/Canny para realce de bordas
→ YOLO para detecção
→ U-Net para segmentação
→ validação técnica
→ relatório
```

Sobel e Canny podem ser usados como etapa didática, exploratória ou auxiliar antes de modelos supervisionados.